# **New approaches to CATE: calibration, conformal intervals, model selection and CausalPFN**

Practice for the module [“Heterogeneous effects”](https://ai-interpretability.school), lesson “How to read the estimate and where to go wrong”, step “What is used today”.

In the step “What is used today” four recent directions were rows of a table. Here we apply all four to one data set with a known effect and check what each one promises and where it stops working:

1. causal isotonic calibration ([van der Laan, Ulloa-Pérez, Carone and Luedtke, 2023](https://arxiv.org/abs/2302.14011)) fixes the numbers $\hat\tau(x)$ without changing the order of objects;
2. weighted conformal intervals ([Lei and Candès, 2021](https://arxiv.org/abs/2006.06138)) give an interval for an unobserved outcome and for the effect of a single object, with coverage on average over objects;
3. CausalPFN ([Balazadeh et al., 2025](https://arxiv.org/abs/2506.07918)) estimates the effect with a transformer trained in advance, without training on our data;
4. the R-score ([Nie and Wager, 2017](https://arxiv.org/abs/1712.04912); implemented as `econml.score.RScorer`) picks a CATE estimate without the true effect — one of the classes of metrics from the reviews by [Curth and van der Schaar (2023)](https://arxiv.org/abs/2302.02923) and [Mahajan et al. (2024)](https://arxiv.org/abs/2211.01939).

Plan of the notebook:

1. data: an observational study with confounding and a known $\tau(x)$;
2. four CATE estimates: the S-learner on boosting and on the lasso, the T-learner, the causal forest;
3. causal isotonic calibration;
4. weighted conformal intervals for $Y(1)$, $Y(0)$ and for the effect of an object;
5. CausalPFN;
6. model selection by the R-score;
7. where it breaks: the R-score on a small sample and CausalPFN with a hidden confounder;
8. implementations.

The notebook runs on a CPU; the whole run takes one to two minutes. The CausalPFN weights (75 MB) are downloaded from Hugging Face on the first call.

In [ ]:
!pip install -q econml==0.17.0 causalpfn==0.1.5

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
from econml.score import RScorer
from causalpfn import CATEEstimator

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some processors
SEED = 0
torch.set_num_threads(1)     # one thread: the same order of additions in CausalPFN on every run
pd.set_option('display.width', 140)

## 1. Data

An observational study: 7000 objects, five features $X\sim\mathcal N(0, I)$. The treatment is not assigned at random: the propensity $e(x)=\sigma(0.6\,x_1-0.3\,x_2)$ ranges from 0.09 to 0.93, and $x_1$ also enters the outcome without treatment $\mu_0(x)=2x_1+x_2^2$ — it is a confounder. The effect is $\tau(x)=1+1.5\,x_0+0.5\max(x_3, 0)$. The outcome noise depends on $x_0$: its standard deviation is $0.5+0.5|x_0|$, so that the intervals for objects with a large $|x_0|$ are wider. Both potential outcomes are generated with independent noise, so an object's effect $Y_i(1)-Y_i(0)$ is not the same as $\tau(x_i)$: it also contains the noise of the two outcomes.

Three parts: on the first 3000 objects (`tr`) the CATE estimates and the quantile models are trained, on the next 2000 (`ca`) the calibration, the conformal correction and the model selection are done, and on the last 2000 (`te`) everything is checked against the true $\tau(x)$, $Y(1)$ and $Y(0)$. A real researcher sees only $X$, $T$ and $Y$.

In [ ]:
def simulate(n, r):
    X = r.normal(size=(n, 5))
    e = 1 / (1 + np.exp(-(0.6 * X[:, 1] - 0.3 * X[:, 2])))
    T = r.binomial(1, e)
    tau = 1 + 1.5 * X[:, 0] + 0.5 * np.maximum(X[:, 3], 0)
    sd = 0.5 + 0.5 * np.abs(X[:, 0])
    mu0 = 2 * X[:, 1] + X[:, 2] ** 2
    Y0 = mu0 + sd * r.normal(size=n)
    Y1 = mu0 + tau + sd * r.normal(size=n)
    return X, T, np.where(T == 1, Y1, Y0), tau, e, Y0, Y1


X, T, Y, tau, e, Y0, Y1 = simulate(7000, np.random.default_rng(SEED))
tr, ca, te = np.arange(3000), np.arange(3000, 5000), np.arange(5000, 7000)
naive = Y[te][T[te] == 1].mean() - Y[te][T[te] == 0].mean()
print('share treated among the 7000 objects:', round(T.mean(), 3), '| propensity from', round(e.min(), 3), 'to', round(e.max(), 3))
print('true average effect on te:', round(tau[te].mean(), 3), '| naive difference of means:', round(naive, 3))

## 2. Four CATE estimates

We take four estimates that have already appeared in the module and spoil one of them on purpose:

- the S-learner on gradient boosting: one outcome model, the treatment is a feature;
- the S-learner on the lasso with interactions $x\cdot t$ and strong regularization (`alpha=0.3`): the penalty pulls the coefficients of $t$ and $x\cdot t$ toward zero, and the effect comes out smaller than the real one — the very failure of the S-learner from the lesson “Metalearners: S, T and X”;
- the T-learner on gradient boosting: one model per group;
- the causal forest `CausalForestDML`.

We measure the error as in the other notebooks of the module: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ on `te`. Next to it is the calibration error: we split the objects into 20 equal groups by the prediction $\hat\tau$ and add up the squared differences between the mean true and the mean predicted effect in each group, weighted by the group's share. This estimates the quantity $\mathrm{CAL}(\hat\tau)=\mathbb E\bigl[(\mathbb E[\tau(X)\mid\hat\tau(X)]-\hat\tau(X))^2\bigr]$ from the paper by van der Laan et al.; it can only be computed when $\tau$ is known.

In [ ]:
def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def gbc():
    return GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)


def pehe(p, t):
    return np.sqrt(np.mean((p - t) ** 2))


def cal_error(p, t, k=20):
    groups = np.array_split(np.argsort(p, kind='stable'), k)
    return sum(len(g) / len(p) * (t[g].mean() - p[g].mean()) ** 2 for g in groups)


def inter(Z, t):
    return np.c_[Z, t, Z * t[:, None]]


s_gb = gb().fit(np.c_[X[tr], T[tr]], Y[tr])
s_lasso = Lasso(alpha=0.3).fit(inter(X[tr], T[tr]), Y[tr])
m1 = gb().fit(X[tr][T[tr] == 1], Y[tr][T[tr] == 1])
m0 = gb().fit(X[tr][T[tr] == 0], Y[tr][T[tr] == 0])
cf = CausalForestDML(model_y=gb(), model_t=gbc(), discrete_treatment=True, n_estimators=500,
                     min_samples_leaf=40, random_state=SEED).fit(Y[tr], T[tr], X=X[tr])
one, zero = lambda Z: np.ones(len(Z)), lambda Z: np.zeros(len(Z))
models = {
    'S-learner, GB': lambda Z: s_gb.predict(np.c_[Z, one(Z)]) - s_gb.predict(np.c_[Z, zero(Z)]),
    'S-learner, lasso': lambda Z: s_lasso.predict(inter(Z, one(Z))) - s_lasso.predict(inter(Z, zero(Z))),
    'T-learner, GB': lambda Z: m1.predict(Z) - m0.predict(Z),
    'causal forest': lambda Z: cf.effect(Z),
}
pred = {k: {'ca': f(X[ca]), 'te': f(X[te])} for k, f in models.items()}
print(pd.DataFrame({k: [pehe(p['te'], tau[te]), cal_error(p['te'], tau[te]), p['te'].mean() - tau[te].mean()] for k, p in pred.items()},
                   index=['PEHE', 'CAL', 'ATE error']).T.round(3))

## 3. Causal isotonic calibration

A prediction $\hat\tau$ is calibrated if, among the objects with prediction $v$, the mean true effect equals $v$. The true effect is not available, but it has an unbiased substitute — the doubly robust pseudo-outcome of the DR-learner:
$$\hat\varphi_i=\hat\mu_1(x_i)-\hat\mu_0(x_i)+\frac{t_i-\hat e(x_i)}{\hat e(x_i)(1-\hat e(x_i))}\bigl(y_i-\hat\mu_{t_i}(x_i)\bigr).$$
Algorithm 1 of the paper by van der Laan et al.: compute $\hat\varphi_i$ on the calibration sample, with the outcome and propensity models taken from other folds of that sample (the cross-fitted variant, Algorithm 4 of the paper), and find the nondecreasing function $\theta$ closest to the pseudo-outcomes in squared error:
$$\hat\theta=\arg\min_{\theta\ \text{nondecreasing}}\ \sum_{i\in\mathrm{ca}}\bigl(\hat\varphi_i-\theta(\hat\tau(x_i))\bigr)^2 .$$
The calibrated prediction is $\hat\theta(\hat\tau(x))$. This is ordinary isotonic regression `sklearn.isotonic.IsotonicRegression` of the pseudo-outcome on the prediction. The function is nondecreasing, so the order of objects is preserved, but neighboring predictions can receive the same value.

In [ ]:
def pseudo_outcome(Xc, Tc, Yc, folds=5):
    phi = np.zeros(len(Yc))
    for a, b in KFold(folds, shuffle=True, random_state=SEED).split(Xc):
        g1 = gb().fit(Xc[a][Tc[a] == 1], Yc[a][Tc[a] == 1]).predict(Xc[b])
        g0 = gb().fit(Xc[a][Tc[a] == 0], Yc[a][Tc[a] == 0]).predict(Xc[b])
        p = np.clip(gbc().fit(Xc[a], Tc[a]).predict_proba(Xc[b])[:, 1], 0.01, 0.99)
        phi[b] = g1 - g0 + (Tc[b] - p) / (p * (1 - p)) * (Yc[b] - np.where(Tc[b] == 1, g1, g0))
    return phi


phi = pseudo_outcome(X[ca], T[ca], Y[ca])
print('mean pseudo-outcome on ca:', round(phi.mean(), 3), '| true average effect on ca:', round(tau[ca].mean(), 3))

**Task 1. Write a function `calibrate(p_ca, p_te)`: isotonic regression of the pseudo-outcome `phi` on the prediction `p_ca` on the sample `ca` (`IsotonicRegression(out_of_bounds='clip')`), returning the calibrated prediction for `p_te`. Calibrate the S-learner on the lasso. What are the PEHE and the CAL before and after calibration, and the mean effect before and after? Plot the prediction against the true $\tau$ before and after.**

In [ ]:
# Your code here

**Your answer:**

**Task 2. Calibrate the other three estimates with the same function. Print a table: the PEHE and the CAL before and after calibration and the number of distinct values of the calibrated prediction. For which estimates does the CAL fall, and for which does the PEHE grow? How does this agree with the decomposition $\mathrm{MSE}=\mathrm{CAL}+\mathrm{DIS}$ from the paper?**

In [ ]:
# Your code here

**Your answer:**

## 4. Weighted conformal intervals

The outcome $Y(1)$ is known only for the treated, but an interval for it is needed for everyone. This is a problem with a shift in the feature distribution: the model is trained on $P_{X\mid T=1}$ and applied on $P_X$. Lei and Candès build the interval with weighted split-CQR (Algorithm 1 of the paper):

1. on the training part of the group $T=1$ fit two outcome quantile models, $\hat q_{\alpha/2}(x)$ and $\hat q_{1-\alpha/2}(x)$;
2. on the calibration part of the group $T=1$ compute the nonconformity score $V_i=\max\{\hat q_{\alpha/2}(x_i)-y_i,\ y_i-\hat q_{1-\alpha/2}(x_i)\}$;
3. give each calibration object the weight $w_1(x_i)=1/\hat e(x_i)$ (Table 1 of the paper: the target is the whole population), and a new object $x$ the weight $w_1(x)$;
4. $\eta(x)$ is the $1-\alpha$ quantile of the distribution that puts mass $w_1(x_i)/(\sum_j w_1(x_j)+w_1(x))$ on $V_i$ and the remainder $w_1(x)/(\sum_j w_1(x_j)+w_1(x))$ on $+\infty$;
5. the interval is $[\hat q_{\alpha/2}(x)-\eta(x),\ \hat q_{1-\alpha/2}(x)+\eta(x)]$.

For $Y(0)$ the same is done on the control group with the weight $w_0(x)=1/(1-\hat e(x))$. Without weights (all $w=1$) this is ordinary split-CQR. With correct weights the coverage is at least $1-\alpha$ on average over objects, but not for every $x$.

In Section 4.1 of the paper the interval for an object's effect $Y(1)-Y(0)$ is built “naively”: intervals for $Y(1)$ and $Y(0)$ at level $1-\alpha/2$ each, and the difference of their ends, $[\hat Y^L(1)-\hat Y^R(0),\ \hat Y^R(1)-\hat Y^L(0)]$. Coverage of at least $1-\alpha$ is guaranteed, but with a margin; the nested scheme from Section 4.2 gives narrower intervals.

In [ ]:
propensity = gbc().fit(X[tr], T[tr])


def e_hat(Z):
    return propensity.predict_proba(Z)[:, 1]


def quantile_gb(q):
    return GradientBoostingRegressor(loss='quantile', alpha=q, n_estimators=200, max_depth=3, random_state=SEED)

**Task 3. Write a function `conformal(t, alpha, weighted)`: it fits the quantile models `quantile_gb` on the objects `tr` of group `t`, computes the nonconformity scores on the objects `ca` of the same group, and returns a function that takes a feature matrix and gives the lower and upper ends of the intervals (with weights $1/\hat e$ or $1/(1-\hat e)$ when `weighted=True`, without weights when `weighted=False`). Build 90 percent intervals for $Y(1)$ and $Y(0)$ on `te` without and with weights, and print the share of objects whose interval covered the true potential outcome — among all objects and among those for whom this outcome was not observed. Then build the naive 90 percent interval for $Y(1)-Y(0)$ with weights: what are its share of covered objects and its mean interval width?**

In [ ]:
# Your code here

**Your answer:**

**Task 4. Coverage on average is not coverage for every object. Split `te` into five equal groups by $x_0$ (`np.argsort(X[te, 0])`, `np.array_split(..., 5)`): the first group is the 400 objects with the smallest $x_0$. For the weighted 90 percent intervals for $Y(1)$, print the share of covered objects and the mean interval width in each group. How many of the 400 objects of the first group are covered, and why is the coverage below 90% exactly there?**

In [ ]:
# Your code here

**Your answer:**

## 5. CausalPFN

CausalPFN is a transformer that the authors trained once on a large library of simulated processes with a known effect, where the no-hidden-confounders assumption holds. It does not train on new data: the table `tr` with $X$, $T$, $Y$ is given to it as context, and it outputs an effect estimate for new objects, the way a prior-fitted network outputs a prediction. `CATEEstimator.fit` only stores the context and loads the weights; the work happens in `estimate_cate`.

The model and the code are under the Apache-2.0 license. The Hugging Face warning about requests without a token can be ignored: the weights are public.

**Task 5. Estimate CATE with CausalPFN (`CATEEstimator(device='cpu')`, the context is `tr`) on `ca` and on `te`. What are the PEHE, the CAL and the error of the average effect on `te`, and how many seconds did the estimation take? Compare with the four estimates from section 2.**

In [ ]:
# Your code here

**Your answer:**

## 6. Model selection by the R-score

On real data the PEHE cannot be computed. The surrogate metric R-score is built on the loss of the R-learner: on the selection sample `ca`, $\hat m(x)=\mathbb E[Y\mid X=x]$ and $\hat e(x)$ are estimated with cross-fitting, and an estimate $\hat\tau$ gets the loss $\frac1n\sum_i\bigl((y_i-\hat m(x_i))-\hat\tau(x_i)(t_i-\hat e(x_i))\bigr)^2$. `econml.score.RScorer` compares it with the loss of the best constant effect: $\text{R-score}=1-\text{loss}(\hat\tau)/\text{loss}(\text{const})$; larger is better, and a negative value is worse than a constant effect. Curth and van der Schaar place such metrics in the “pseudo-outcome” class and show that any surrogate metric favors estimates that resemble its own auxiliary models.

`RScorer.score` expects an object with the method `const_marginal_effect`, so we wrap the ready predictions in a small class.

In [ ]:
class Fixed:
    # ready predictions on the rows of ca in the form RScorer understands
    def __init__(self, values):
        self.values = values

    def const_marginal_effect(self, Z):
        return self.values

**Task 6. Fit `RScorer(model_y=gb(), model_t=gbc(), discrete_treatment=True, cv=5, random_state=SEED)` on `ca` and compute the R-score of the five estimates (the four from section 2 and CausalPFN). Print a table of the R-score and the PEHE on `te` and the Spearman rank correlation between the R-score and $-\mathrm{PEHE}$. Which estimate does the R-score pick, and does its order agree with the order by PEHE?**

In [ ]:
# Your code here

**Your answer:**

## 7. Where it breaks

### 7.1. The R-score on a small sample

The R-score is also an estimate from a sample, with its own noise: both the auxiliary models $\hat m$, $\hat e$ and the loss itself depend on which objects fell into the selection sample. In section 6 the R-score had 2000 objects. Let us see what it does on five disjoint parts of 400.

**Task 7. Split `ca` into five parts of 400 (`np.array_split(np.random.default_rng(SEED).permutation(len(ca)), 5)` — positions within `ca`), fit the same `RScorer` on each and compute the R-score of the five estimates. Which estimate does the R-score pick on each part? On how many parts did the choice agree with the choice on the whole `ca`, and how close are the R-scores of the first and second estimates there?**

In [ ]:
# Your code here

**Your answer:**

### 7.2. CausalPFN and a hidden confounder

The guarantees of CausalPFN are derived under the assumption that all confounders are among the features: the model was trained on such processes. Let us check what happens if the confounder $x_1$ is hidden — removed from the features, as if it had not been recorded. Its influence on the treatment and on the outcome stays.

**Task 8. Remove $x_1$ from the features (`keep = [0, 2, 3, 4]`) and estimate CATE on `te` with CausalPFN (the context is `tr` without $x_1$) and with the causal forest from section 2 trained without $x_1$. What are the PEHE and the error of the average effect for both? Compare with the naive difference of means from section 1.**

In [ ]:
# Your code here

**Your answer:**

## 8. Implementations

- Causal isotonic calibration: the authors' R package [causalCalibration](https://github.com/Larsvanderlaan/causalCalibration) (MIT license). In Python — as here: the DR-learner pseudo-outcome and `sklearn.isotonic.IsotonicRegression`; the paper also has cross-calibration, which calibrates a prediction obtained by cross-fitting on all the data, without a separate calibration sample.
- Conformal intervals for potential outcomes and ITE: the authors' R package [cfcausal](https://github.com/lihualei71/cfcausal) (MIT license) — weighted CQR, the naive and the nested schemes. There is no ready Python implementation for the effect; weighted split-CQR is a 15-line function, as above.
- Model selection: [`econml.score.RScorer`](https://github.com/py-why/EconML) (MIT license) — the R-score, plus `RScorer.best_model` and `RScorer.ensemble` for picking and mixing estimates.
- CausalPFN: [`causalpfn`](https://github.com/vdblm/CausalPFN) (Apache-2.0 license): `CATEEstimator` and `ATEEstimator`; `CATEEstimator` also has uncertainty calibration (`calibrate=True`) and intervals `estimate_cate_CI`.

## What to take away

- **Calibration fixes the numbers, not the order.** The S-learner spoiled by regularization went, after isotonic calibration on the pseudo-outcome, from wrong in level to one of the best; for estimates that were already nearly calibrated the CAL falls and the PEHE grows a little: the calibrated prediction takes only 68–83 distinct values.
- **A conformal interval covers on average.** Propensity weighting carries the coverage from the observed group over to the whole population, but in the region with large noise the share covered is below the stated level; the naive interval for an object's effect is reliable and wide.
- **CausalPFN, without training on our data, gave the smallest PEHE** — with the no-hidden-confounders assumption holding. Without the confounder among the features it errs just as the causal forest does: neither estimate checks the assumption.
- **The R-score on 2000 objects ordered the estimates the same way as the PEHE,** but on 400 objects the choice is unstable where the R-scores of two CATE estimates differ in the third or fourth decimal.

Sources:
1. [van der Laan, Ulloa-Pérez, Carone & Luedtke, 2023. Causal isotonic calibration for heterogeneous treatment effects](https://arxiv.org/abs/2302.14011)
2. [Lei & Candès, 2021. Conformal inference of counterfactuals and individual treatment effects](https://arxiv.org/abs/2006.06138)
3. [Nie & Wager, 2017. Quasi-oracle estimation of heterogeneous treatment effects](https://arxiv.org/abs/1712.04912)
4. [Curth & van der Schaar, 2023. In search of insights, not magic bullets: towards demystification of the model selection dilemma in heterogeneous treatment effect estimation](https://arxiv.org/abs/2302.02923)
5. [Mahajan et al., 2024. Empirical analysis of model selection for heterogeneous causal effect estimation](https://arxiv.org/abs/2211.01939)
6. [Balazadeh et al., 2025. CausalPFN: Amortized causal effect estimation via in-context learning](https://arxiv.org/abs/2506.07918)